In [ ]:
!pip install -q FlagEmbedding qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.4 MB/s eta 0:00:00


In [ ]:
from google.colab import files
print("Upload dtc_chunks.jsonl, warranty_chunks.jsonl, and nhtsa_chunks.jsonl")
uploaded = files.upload()

Upload dtc_chunks.jsonl, warranty_chunks.jsonl, and nhtsa_chunks.jsonl


Saving warranty_chunks.jsonl to warranty_chunks (1).jsonl
Saving nhtsa_chunks.jsonl to nhtsa_chunks (1).jsonl
Saving dtc_chunks.jsonl to dtc_chunks (1).jsonl


In [ ]:
import json

FILES = ["dtc_chunks.jsonl", "warranty_chunks.jsonl", "nhtsa_chunks.jsonl"]

all_chunks = []
for fname in FILES:
    with open(fname, "r", encoding="utf-8") as f:
        for line in f:
            all_chunks.append(json.loads(line))

print(f"Total chunks loaded: {len(all_chunks)}")

Total chunks loaded: 59089


In [ ]:
from FlagEmbedding import BGEM3FlagModel
import numpy as np
import time
import pickle

print("Loading BGE-M3 model (GPU)...")
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print("Model loaded.")

texts = [chunk["text"] for chunk in all_chunks]

BATCH_SIZE = 128  # GPU can handle much bigger batches than your CPU could
CHECKPOINT_EVERY = 20  # save progress every 20 batches, in case Colab disconnects

dense_vecs_all = []
sparse_vecs_all = []

start = time.time()
num_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(texts), BATCH_SIZE):
    batch_num = i // BATCH_SIZE + 1
    batch_texts = texts[i:i + BATCH_SIZE]

    output = model.encode(
        batch_texts,
        batch_size=BATCH_SIZE,
        max_length=512,
        return_dense=True,
        return_sparse=True,
    )

    dense_vecs_all.append(output["dense_vecs"])
    sparse_vecs_all.extend(output["lexical_weights"])

    if batch_num % CHECKPOINT_EVERY == 0 or batch_num == num_batches:
        elapsed = time.time() - start
        done = min(i + BATCH_SIZE, len(texts))
        print(f"Batch {batch_num}/{num_batches} | {done}/{len(texts)} chunks | {elapsed/60:.1f} min elapsed")

        # Save checkpoint to disk
        with open("embeddings_checkpoint.pkl", "wb") as f:
            pickle.dump({
                "dense_vecs": np.vstack(dense_vecs_all),
                "sparse_vecs": sparse_vecs_all,
                "num_done": done,
            }, f)

dense_vecs_final = np.vstack(dense_vecs_all)
print(f"\nDone. Total time: {(time.time()-start)/60:.1f} minutes")
print(f"Dense vectors shape: {dense_vecs_final.shape}")

Loading BGE-M3 model (GPU)...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Batch 440/462 | 56320/59089 chunks | 33.1 min elapsed


Inference Embeddings: 100%|██████████| 1/1 [00:04<00:00,  4.46s/it]


Batch 460/462 | 58880/59089 chunks | 34.9 min elapsed


Inference Embeddings: 100%|██████████| 1/1 [00:04<00:00,  4.55s/it]


Batch 462/462 | 59089/59089 chunks | 35.3 min elapsed

Done. Total time: 35.5 minutes
Dense vectors shape: (59089, 1024)


In [ ]:
from google.colab import userdata
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, SparseVectorParams, SparseVector
import uuid

QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

COLLECTION_NAME = "autodiag_rag"

# Create collection with both dense and sparse vector support (hybrid!)
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(size=1024, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)
print(f"Collection '{COLLECTION_NAME}' created.")

# Build points: each point = one chunk with dense + sparse vectors + metadata
def sparse_dict_to_qdrant(lexical_weights):
    """Convert BGE-M3's {token_id: weight} dict into Qdrant's SparseVector format."""
    indices = [int(k) for k in lexical_weights.keys()]
    values = [float(v) for v in lexical_weights.values()]
    return SparseVector(indices=indices, values=values)

BATCH_SIZE = 256
total = len(all_chunks)

for start_idx in range(0, total, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total)
    points = []
    for i in range(start_idx, end_idx):
        chunk = all_chunks[i]
        points.append(PointStruct(
            id=str(uuid.uuid4()),
            vector={
                "dense": dense_vecs_final[i].tolist(),
                "sparse": sparse_dict_to_qdrant(sparse_vecs_all[i]),
            },
            payload={
                "text": chunk["text"],
                **chunk["metadata"],
            }
        ))
    client.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"Uploaded {end_idx}/{total}")

print("\nAll chunks uploaded to Qdrant Cloud.")

# Verify
info = client.get_collection(COLLECTION_NAME)
print(f"Collection point count: {info.points_count}")

/tmp/ipykernel_1196/223140599.py:14: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Collection 'autodiag_rag' created.
Uploaded 256/59089
Uploaded 512/59089
Uploaded 768/59089
Uploaded 1024/59089
Uploaded 1280/59089
Uploaded 1536/59089
Uploaded 1792/59089
Uploaded 2048/59089
Uploaded 2304/59089
Uploaded 2560/59089
Uploaded 2816/59089
Uploaded 3072/59089
Uploaded 3328/59089
Uploaded 3584/59089
Uploaded 3840/59089
Uploaded 4096/59089
Uploaded 4352/59089
Uploaded 4608/59089
Uploaded 4864/59089
Uploaded 5120/59089
Uploaded 5376/59089
Uploaded 5632/59089
Uploaded 5888/59089
Uploaded 6144/59089
Uploaded 6400/59089
Uploaded 6656/59089
Uploaded 6912/59089
Uploaded 7168/59089
Uploaded 7424/59089
Uploaded 7680/59089
Uploaded 7936/59089
Uploaded 8192/59089
Uploaded 8448/59089
Uploaded 8704/59089
Uploaded 8960/59089
Uploaded 9216/59089
Uploaded 9472/59089
Uploaded 9728/59089
Uploaded 9984/59089
Uploaded 10240/59089
Uploaded 10496/59089
Uploaded 10752/59089
Uploaded 11008/59089
Uploaded 11264/59089
Uploaded 11520/59089
Uploaded 11776/59089
Uploaded 12032/59089
Uploaded 12288/59089